# Jakarta Globe News Scraper - Jupyter Notebook

This notebook demonstrates web scraping of Jakarta Globe news articles and converts the results to pandas DataFrame format.

In [1]:
# Install required packages
!pip install beautifulsoup4 lxml requests pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime
import time

In [3]:
# Configuration
BASE_URL = "https://jakartaglobe.id/search/business/"
MAX_PAGES = 5  # Number of pages to scrape
REQUEST_DELAY = 1  # Delay between requests (seconds)

In [4]:
def get_page_url(page: int) -> str:
    """Generate URL for specific page."""
    if page == 1:
        return f"{BASE_URL}1"
    return f"{BASE_URL}{page}"

def fetch_page(url: str) -> str:
    """Fetch HTML content from URL."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None

def parse_article(elem) -> dict:
    """Parse article from HTML element."""
    article = {}
    
    # Try to find title and URL
    if elem.name == 'a':
        article['url'] = elem.get('href', '')
        title_elem = elem.select_one('h2, h3, h4, .title, .article-title')
        article['title'] = title_elem.get_text(strip=True) if title_elem else elem.get_text(strip=True)
    else:
        link_elem = elem.select_one('a[href*="/read/"]')
        if not link_elem:
            return None
        
        article['url'] = link_elem.get('href', '')
        title_elem = (elem.select_one('h2, h3, h4, .title, .article-title, .news-title') or 
                      link_elem.select_one('h2, h3, h4, .title'))
        article['title'] = title_elem.get_text(strip=True) if title_elem else link_elem.get_text(strip=True)
    
    # Validate URL
    if not article.get('url') or '/read/' not in article['url']:
        return None
    
    # Make URL absolute
    if not article['url'].startswith('http'):
        article['url'] = 'https://jakartaglobe.id' + article['url']
    
    # Extract date from URL
    date_match = re.search(r'/read/(\d{4})(\d{2})(\d{2})/', article['url'])
    if date_match:
        year, month, day = date_match.groups()
        article['date'] = f"{year}-{month}-{day}"
    else:
        article['date'] = datetime.now().strftime('%Y-%m-%d')
    
    # Extract category
    category_elem = elem.select_one('.category, .category-tag, span.category')
    article['category'] = category_elem.get_text(strip=True) if category_elem else 'Business'
    
    return article

def scrape_listing_page(page: int) -> list:
    """Scrape articles from a listing page."""
    url = get_page_url(page)
    print(f"Scraping page {page}: {url}")
    
    html = fetch_page(url)
    if not html:
        return []
    
    soup = BeautifulSoup(html, 'lxml')
    articles = []
    
    # Find article elements - try multiple selectors
    article_elements = (
        soup.select('div.article-item') or
        soup.select('div.news-item') or
        soup.select('div.article-card') or
        soup.select('article') or
        soup.select('div.latest-news article') or
        soup.select('div.search-result article') or
        soup.select('div.news-list article') or
        []
    )
    
    if not article_elements:
        # Try finding links that look like articles
        article_elements = soup.select('a[href*="/read/"]')
    
    for elem in article_elements:
        article = parse_article(elem)
        if article:
            articles.append(article)
    
    print(f"Found {len(articles)} articles on page {page}")
    return articles

In [5]:
# Scrape articles from multiple pages
all_articles = []

for page in range(1, MAX_PAGES + 1):
    articles = scrape_listing_page(page)
    all_articles.extend(articles)
    time.sleep(REQUEST_DELAY)

print(f"\nTotal articles scraped: {len(all_articles)}")

Scraping page 1: https://jakartaglobe.id/search/business/1
Error fetching https://jakartaglobe.id/search/business/1: 403 Client Error: Forbidden for url: https://jakartaglobe.id/search/business/1
Scraping page 2: https://jakartaglobe.id/search/business/2
Error fetching https://jakartaglobe.id/search/business/2: 403 Client Error: Forbidden for url: https://jakartaglobe.id/search/business/2
Scraping page 3: https://jakartaglobe.id/search/business/3
Error fetching https://jakartaglobe.id/search/business/3: 403 Client Error: Forbidden for url: https://jakartaglobe.id/search/business/3
Scraping page 4: https://jakartaglobe.id/search/business/4
Error fetching https://jakartaglobe.id/search/business/4: 403 Client Error: Forbidden for url: https://jakartaglobe.id/search/business/4
Scraping page 5: https://jakartaglobe.id/search/business/5
Error fetching https://jakartaglobe.id/search/business/5: 403 Client Error: Forbidden for url: https://jakartaglobe.id/search/business/5

Total articles scra

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(all_articles)

# Display basic info
print("DataFrame shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
# Display the DataFrame
df

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Display article titles
print("Article Titles:\n")
for i, title in enumerate(df['title'].head(10), 1):
    print(f"{i}. {title}")

In [ ]:
# Show date distribution
print("Date distribution:")
print(df['date'].value_counts())

In [ ]:
# Show category distribution
print("Category distribution:")
print(df['category'].value_counts())

In [ ]:
# Save to CSV
df.to_csv('jktglobe_news.csv', index=False)
print("Data saved to jktglobe_news.csv")

In [ ]:
# Save to JSON
df.to_json('jktglobe_news.json', orient='records', indent=2)
print("Data saved to jktglobe_news.json")

## Extended: Fetch Article Content

The following code fetches full content from individual articles.

In [ ]:
def fetch_article_content(url: str) -> dict:
    """Fetch full article content."""
    content_data = {
        'content': '',
        'author': '',
        'published_time': '',
        'summary': ''
    }
    
    html = fetch_page(url)
    if not html:
        return content_data
    
    soup = BeautifulSoup(html, 'lxml')
    
    # Extract content
    content_elem = (
        soup.select_one('div.article-content') or
        soup.select_one('div.content') or
        soup.select_one('div.article-body') or
        soup.select_one('div.post-content') or
        soup.select_one('article div.content') or
        soup.select_one('div.entry-content')
    )
    
    if content_elem:
        paragraphs = content_elem.select('p')
        content_data['content'] = '\n\n'.join(
            p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
        )
        if paragraphs and paragraphs[0].get_text(strip=True):
            content_data['summary'] = paragraphs[0].get_text(strip=True)
    
    # Extract author
    author_elem = (
        soup.select_one('meta[name="author"]') or
        soup.select_one('.author-name') or
        soup.select_one('.author')
    )
    if author_elem:
        content_data['author'] = author_elem.get('content', '') or author_elem.get_text(strip=True)
    
    # Extract published time
    published_elem = soup.select_one('meta[property="article:published_time"]')
    if published_elem:
        content_data['published_time'] = published_elem.get('content', '')
    
    return content_data

In [ ]:
# Fetch content for first 5 articles (demo)
articles_with_content = []

for i, article in enumerate(all_articles[:5]):
    print(f"Fetching content {i+1}/5: {article.get('title', '')[:50]}...")
    content_data = fetch_article_content(article['url'])
    article.update(content_data)
    articles_with_content.append(article)
    time.sleep(REQUEST_DELAY)

print("\nContent fetched for articles!")

In [ ]:
# Convert to DataFrame with content
df_content = pd.DataFrame(articles_with_content)
df_content

In [ ]:
# Display summary for each article
for i, row in df_content.iterrows():
    print(f"\n{'='*60}")
    print(f"Title: {row['title']}")
    print(f"URL: {row['url']}")
    print(f"Date: {row['date']}")
    print(f"Author: {row.get('author', 'N/A')}")
    print(f"Summary: {row.get('summary', 'N/A')[:200]}...")
    print(f"Content length: {len(row.get('content', ''))} chars")